# Assignment 4

In [1]:
import networkx as nx
import pandas as pd
import numpy as np
import pickle

---

## Part 1 - Random Graph Identification

For the first part of this assignment you will analyze randomly generated graphs and determine which algorithm created them.

In [2]:
G1 = nx.read_gpickle("assets/A4_P1_G1")
G2 = nx.read_gpickle("assets/A4_P1_G2")
G3 = nx.read_gpickle("assets/A4_P1_G3")
G4 = nx.read_gpickle("assets/A4_P1_G4")
G5 = nx.read_gpickle("assets/A4_P1_G5")
P1_Graphs = [G1, G2, G3, G4, G5]

<br>
`P1_Graphs` is a list containing 5 networkx graphs. Each of these graphs were generated by one of three possible algorithms:
* Preferential Attachment (`'PA'`)
* Small World with low probability of rewiring (`'SW_L'`)
* Small World with high probability of rewiring (`'SW_H'`)

Anaylze each of the 5 graphs using any methodology and determine which of the three algorithms generated each graph.

*The `graph_identification` function should return a list of length 5 where each element in the list is either `'PA'`, `'SW_L'`, or `'SW_H'`.*

In [15]:
def graph_identification():
    # YOUR CODE HERE
    #raise NotImplementedError()
    
    #* Preferential Attachment (`'PA'`): 
    #1. produces networks with small shortest paths but very small clustering coefficient. 
    #2. the degree distribution of small world networks is a power law.
    
    #* Small World with low probability of rewiring (`'SW_L'`) :  
    #1. the degree distribution of small world networks is not a power law.
    #2. small shortest paths between nodes and high clustering coefficient.
    
    #* Small World with high probability of rewiring (`'SW_H'`) :  
    #1. the degree distribution of small world networks is not a power law.
    #2. small shortest paths between nodes and high clustering coefficient.
    #3. average shortest path decreases rapidly and average clustering coefficient deceases slowly. 
    
    #for grph in P1_Graphs:
        #print("avg_clustering_coeff:", nx.average_clustering(grph))
        #print("avg_shrtest_path:", nx.average_shortest_path_length(grph))
        #print('\n')
    
    return ['PA', 'SW_L', 'SW_L','PA', 'SW_H']

graph_identification()

['PA', 'SW_L', 'SW_L', 'PA', 'SW_H']

In [16]:
ans_one = graph_identification()
assert type(ans_one) == list, "You must return a list"


---

## Part 2 - Company Emails

For the second part of this assignment you will be working with a company's email network where each node corresponds to a person at the company, and each edge indicates that at least one email has been sent between two people.

The network also contains the node attributes `Department` and `ManagmentSalary`.

`Department` indicates the department in the company which the person belongs to, and `ManagmentSalary` indicates whether that person is receiving a managment position salary.

In [2]:
G = pickle.load(open('assets/email_prediction_NEW.txt', 'rb'))

print(f"Graph with {len(nx.nodes(G))} nodes and {len(nx.edges(G))} edges")

Graph with 1005 nodes and 16706 edges


### Part 2A - Salary Prediction

Using network `G`, identify the people in the network with missing values for the node attribute `ManagementSalary` and predict whether or not these individuals are receiving a managment position salary.

To accomplish this, you will need to create a matrix of node features of your choice using networkx, train a sklearn classifier on nodes that have `ManagementSalary` data, and predict a probability of the node receiving a managment salary for nodes where `ManagementSalary` is missing.



Your predictions will need to be given as the probability that the corresponding employee is receiving a managment position salary.

The evaluation metric for this assignment is the Area Under the ROC Curve (AUC).

Your grade will be based on the AUC score computed for your classifier. A model which with an AUC of 0.75 or higher will recieve full points.

Using your trained classifier, return a Pandas series of length 252 with the data being the probability of receiving managment salary, and the index being the node id.

    Example:
    
        1       1.0
        2       0.0
        5       0.8
        8       1.0
            ...
        996     0.7
        1000    0.5
        1001    0.0
        Length: 252, dtype: float64

In [3]:
list(G.nodes(data=True))[:5] # print the first 5 nodes

[(0, {'Department': 1, 'ManagementSalary': 0.0}),
 (1, {'Department': 1, 'ManagementSalary': nan}),
 (581, {'Department': 3, 'ManagementSalary': 0.0}),
 (6, {'Department': 25, 'ManagementSalary': 1.0}),
 (65, {'Department': 4, 'ManagementSalary': nan})]

In [15]:
def salary_predictions():
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import RandomForestClassifier

    # YOUR CODE HERE
    from sklearn.metrics import roc_curve, auc
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    
    from sklearn.svm import SVC
    from sklearn.model_selection import GridSearchCV
    from sklearn.metrics import roc_auc_score
    #raise NotImplementedError()
    
    #is it directed graph: False
    #print(nx.is_directed(G))
    
    # you will need to create a matrix of node features of your choice:
    #degree centrality: important nodes have many connections. 
    degCent=nx.degree_centrality(G) 
    #print(degCent[1])
    
    #closeness centrality: important nodes are close to other nodes
    closeCent=nx.closeness_centrality(G) 
    #print(closeCent[1])
    
    #pagerank: Important nodes are those with many in-links from important pages
    pagerank_scores = nx.pagerank(G, alpha=0.85)
    #print(pagerank_scores[1])
    
    #HITS algorithm: hubs and auth score: Output values for hubs and authorities will be identical per node
    hubs, authorities = nx.hits(G)
    #print(hubs[581])
    
    #load nodes and attributes to data frame
    node_lst = list(G.nodes(data=True))
    
    nd_nm_lst = []
    nd_dpt_lst = []
    nd_mng_sal_lst = []
    
    nd_degcent_lst= []
    nd_clscent_lst= []
    nd_pgrnk_lst= []
    nd_hbs_lst= []
    
    for nde in node_lst:
        nd_nm_lst.append(nde[0])
        nd_dpt_lst.append(nde[1]['Department'])
        nd_mng_sal_lst.append(nde[1]['ManagementSalary'])
        
        nd_degcent_lst.append(degCent[nde[0]])
        nd_clscent_lst.append(closeCent[nde[0]])
        nd_pgrnk_lst.append(pagerank_scores[nde[0]])
        nd_hbs_lst.append(hubs[nde[0]])
        
    df_nds = pd.DataFrame({'Node number': nd_nm_lst, 'Department': nd_dpt_lst, 
                          'Degree Centrality':nd_degcent_lst, 'Closeness Centrality':nd_clscent_lst, 'Pagerank score':nd_pgrnk_lst, 'Hubs':nd_hbs_lst, 
                           'ManagementSalary': nd_mng_sal_lst,})
    #print(df_nds.head())
    
    
    
    #extract test data where management salary is NaN
    nan_rows = df_nds[df_nds['ManagementSalary'].isna()]  
    #print(nan_rows.head())
    non_nan_rows = df_nds[df_nds['ManagementSalary'].notna()] 
    
    #split data into test and train
    
    X_train = non_nan_rows.iloc[:,:-1]
    y_train = non_nan_rows.iloc[:,-1]
    
    X_test = nan_rows.iloc[:,:-1]
    #print(X_test)
    
    lr = LogisticRegression(solver='liblinear').fit(X_train, y_train)
    y_score_lr = lr.predict_proba(X_test) 
    y_score_proba_series = pd.Series(data=y_score_lr[:,1], index=X_test['Node number'])
    
    
    return y_score_proba_series
    #print(list(G.nodes(data=True))[0])
    
salary_predictions()

Node number
1      0.259334
65     0.279592
18     0.259942
215    0.195505
283    0.457629
         ...   
691    0.050001
788    0.072241
944    0.049971
798    0.019346
808    0.036588
Length: 252, dtype: float64

In [16]:
ans_salary_preds = salary_predictions()
assert type(ans_salary_preds) == pd.core.series.Series, "You must return a Pandas series"
assert len(ans_salary_preds) == 252, "The series must be of length 252"


### Part 2B - New Connections Prediction

For the last part of this assignment, you will predict future connections between employees of the network. The future connections information has been loaded into the variable `future_connections`. The index is a tuple indicating a pair of nodes that currently do not have a connection, and the `Future Connection` column indicates if an edge between those two nodes will exist in the future, where a value of 1.0 indicates a future connection.

In [3]:
future_connections = pd.read_csv('assets/Future_Connections.csv', index_col=0, converters={0: eval})
future_connections.head(10)

,Future Connection
"(6, 840)",0.0
"(4, 197)",0.0
"(620, 979)",0.0
"(519, 872)",0.0
"(382, 423)",0.0
"(97, 226)",1.0
"(349, 905)",0.0
"(429, 860)",0.0
"(309, 989)",0.0
"(468, 880)",0.0


Using network `G` and `future_connections`, identify the edges in `future_connections` with missing values and predict whether or not these edges will have a future connection.

To accomplish this, you will need to:      
1. Create a matrix of features of your choice for the edges found in `future_connections` using Networkx     
2. Train a sklearn classifier on those edges in `future_connections` that have `Future Connection` data     
3. Predict a probability of the edge being a future connection for those edges in `future_connections` where `Future Connection` is missing.



Your predictions will need to be given as the probability of the corresponding edge being a future connection.

The evaluation metric for this assignment is the Area Under the ROC Curve (AUC).

Your grade will be based on the AUC score computed for your classifier. A model which with an AUC of 0.75 or higher will recieve full points.

Using your trained classifier, return a series of length 122112 with the data being the probability of the edge being a future connection, and the index being the edge as represented by a tuple of nodes.

    Example:
    
        (107, 348)    0.35
        (542, 751)    0.40
        (20, 426)     0.55
        (50, 989)     0.35
                  ...
        (939, 940)    0.15
        (555, 905)    0.35
        (75, 101)     0.65
        Length: 122112, dtype: float64

In [7]:
def new_connections_predictions():
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.ensemble import RandomForestClassifier

    # YOUR CODE HERE
    from sklearn.metrics import roc_curve, auc
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    
    from sklearn.svm import SVC
    from sklearn.model_selection import GridSearchCV
    from sklearn.metrics import roc_auc_score
    #raise NotImplementedError()
    
    future_connections = pd.read_csv('assets/Future_Connections.csv', index_col=0, converters={0: eval})
    
    #Create a matrix of features of your choice for the edges found in future_connections using Networkx
    
    #common neighbors
    
    #Jaccard Coefficient
    jac_coef_dct={}
    jac_coef_scr=list(nx.jaccard_coefficient(G))
    for cnt_nd in jac_coef_scr:
        jac_coef_dct[cnt_nd[:2]] = cnt_nd[-1]
    
    future_connections['jaccard_coeffecient_scr'] = None
    for idx in future_connections.index:
        if jac_coef_dct[idx]:
            future_connections.at[idx, 'jaccard_coeffecient_scr'] = jac_coef_dct[idx]
    future_connections['jaccard_coeffecient_scr'].fillna(0, inplace=True)
    
    #Resource Allocation
    res_all_scr_dct={}
    res_all_scr=list(nx.resource_allocation_index(G))
    for cnt_nd in res_all_scr:
        res_all_scr_dct[cnt_nd[:2]] = cnt_nd[-1]
    
    future_connections['resource_allocation'] = None
    for idx in future_connections.index:
        if res_all_scr_dct[idx]:
            future_connections.at[idx, 'resource_allocation'] = res_all_scr_dct[idx]
    future_connections['resource_allocation'].fillna(0, inplace=True)
       
    #Adamic-Adar Index
    adam_adr_scr_dct={}
    adam_adr_scr=list(nx.adamic_adar_index(G))
    for cnt_nd in adam_adr_scr:
        adam_adr_scr_dct[cnt_nd[:2]] = cnt_nd[-1]
    
    future_connections['adam_adr_scr'] = None
    for idx in future_connections.index:
        if adam_adr_scr_dct[idx]:
            future_connections.at[idx, 'adam_adr_scr'] = adam_adr_scr_dct[idx]
    
    future_connections['adam_adr_scr'].fillna(0, inplace=True)
    #print(adam_adr_scr[2])
    #print(adam_adr_scr[2][:2])
    #print(adam_adr_scr[2][-1])
    
    #Pref. Attachment  
    pref_att_scr_dct={}
    pref_att_scr=list(nx.preferential_attachment(G))
    for cnt_nd in pref_att_scr:
        pref_att_scr_dct[cnt_nd[:2]] = cnt_nd[-1]
    
    future_connections['preferential_attachment_scr'] = None
    for idx in future_connections.index:
        if pref_att_scr_dct[idx]:
            future_connections.at[idx, 'preferential_attachment_scr'] = pref_att_scr_dct[idx]
    
    future_connections['preferential_attachment_scr'].fillna(0, inplace=True)
    
    #print(pref_att_scr[:2])
    #print(len(list(nx.preferential_attachment(G))))
    
    
    #add scores to future connections df
    #print(future_connections.shape)
    #print(future_connections.index.tolist()[:10])
    
        
    future_connections = future_connections[['jaccard_coeffecient_scr', 'resource_allocation', 'adam_adr_scr','preferential_attachment_scr','Future Connection']]
    #print(future_connections.head())
    
    #extract missing rows for future connections
    nan_rows = future_connections[future_connections['Future Connection'].isna()] 
    #print(nan_rows.shape)
    
    non_nan_rows = future_connections[future_connections['Future Connection'].notna()] 
    #print(non_nan_rows.shape)
    
    #split data into test and train
    
    X_train = non_nan_rows.iloc[:,:-1]
    y_train = non_nan_rows.iloc[:,-1]
    
    X_test = nan_rows.iloc[:,:-1]
    #print(X_test)
    
    lr = LogisticRegression(solver='liblinear').fit(X_train, y_train)
    y_score_lr = lr.predict_proba(X_test) 
    y_score_proba_series = pd.Series(data=y_score_lr[:,1], index=X_test.index)
    
    
    return y_score_proba_series
    
    

new_connections_predictions()

            jaccard_coeffecient_scr  resource_allocation  adam_adr_scr  \
(6, 840)                   0.073770             0.136721      2.110314   
(4, 197)                   0.015504             0.008437      0.363528   
(620, 979)                 0.000000             0.000000      0.000000   
(519, 872)                 0.060606             0.039726      0.507553   
(382, 423)                 0.000000             0.000000      0.000000   

           preferential_attachment_scr  Future Connection  
(6, 840)                          2070                0.0  
(4, 197)                          3552                0.0  
(620, 979)                          28                0.0  
(519, 872)                         299                0.0  
(382, 423)                         205                0.0  


(107, 348)    0.040964
(542, 751)    0.014634
(20, 426)     0.532770
(50, 989)     0.014905
(942, 986)    0.015201
                ...   
(165, 923)    0.012142
(673, 755)    0.015215
(939, 940)    0.015201
(555, 905)    0.014477
(75, 101)     0.022331
Length: 122112, dtype: float64

In [8]:
ans_prob_preds = new_connections_predictions()
assert type(ans_prob_preds) == pd.core.series.Series, "You must return a Pandas series"
assert len(ans_prob_preds) == 122112, "The series must be of length 122112"


            jaccard_coeffecient_scr  resource_allocation  adam_adr_scr  \
(6, 840)                   0.073770             0.136721      2.110314   
(4, 197)                   0.015504             0.008437      0.363528   
(620, 979)                 0.000000             0.000000      0.000000   
(519, 872)                 0.060606             0.039726      0.507553   
(382, 423)                 0.000000             0.000000      0.000000   

           preferential_attachment_scr  Future Connection  
(6, 840)                          2070                0.0  
(4, 197)                          3552                0.0  
(620, 979)                          28                0.0  
(519, 872)                         299                0.0  
(382, 423)                         205                0.0  
